# Cluster Labeling

The purpose of this notebook is to label our clusters with the help of hand-labeled samples (`replay_labels.csv`).
Every labeled replay in a cluster gives votes, and the votes decide the label of the cluster.
Each clustering method has its own execute cell, so they can be run one at a time.

**Index**

- libraries
- settings
- import data
- voting logic
- execute
    - 03b - movement & camera
    - 04b - mcplayhd concept
    - look-up for specific cluster labeling
- old voting logic (unused)

## libraries

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

## settings

In [2]:
DATA_DIR = Path("../data")
EXPORT_DIR = DATA_DIR / "notebooks_exports"

REPLAY_LABELS_PATH = DATA_DIR / "replay_labels.csv"

# method -> clustering result csv written by 03_data-analysis
CLUSTER_EXPORT_PATHS = {
    "movement_camera": EXPORT_DIR / "labeling_cluster_export_movement_camera.csv",
    "mcplayhd_concept": EXPORT_DIR / "labeling_cluster_export_mcplayhd_concept.csv",
}

# method -> where its cluster labels get written
CLUSTER_LABELS_PATHS = {
    "movement_camera": EXPORT_DIR / "cluster_labels_movement_camera.csv",
    "mcplayhd_concept": EXPORT_DIR / "cluster_labels_mcplayhd_concept.csv",
}

# method -> one row per replay with its cluster label and confidence
LABELED_REPLAYS_PATHS = {
    "movement_camera": EXPORT_DIR / "labeled_replays_movement_camera.csv",
    "mcplayhd_concept": EXPORT_DIR / "labeled_replays_mcplayhd_concept.csv",
}

PLAYSTYLE_SEPARATOR = "_"  # splits a playstyle into its parts, each part is one vote
LABEL_JOINER = ", "         # joins the winning parts into the cluster label
CLEAR_SHARE = 0.8
MAJORITY_SHARE = 0.5
MAYBE_SHARE = 0.4
HIGHLY_SUSPICIOUS_THRESHOLD = 1.8
UNCLEAR_LABEL = "unclear"
TOO_SMALL_LABEL = "too_small"
MIN_LABELED_MEMBERS = 3

FANCY_LABELS = {
            "sneak": "sneak bridge",
            "ninja": "ninja bridge",
            "noshift": "noshift staircase",
            "telly": "telly bridge",
            "breezily": "breezily bridge",
            "moon": "moonwalk",
            "god": "god bridge",
            "tower": "towering",
            "0cps": "0cps bridge",
            "cheats": "just cheats",
            "cheats_combined": "cheats with",
            "ac": "just autoclicker",
            "ac_combined": "autoclicker with",
            "bug": "looks buggy",
            "noob": "sth weird",
            "unclear": "no clear technique",
            "stair": "onestack",
            "stair_combined": "(onestack shaped)" # special
        }

## import data

In [3]:
df_replay_labels = pd.read_csv(REPLAY_LABELS_PATH)
# drop unlabeled rows from the label file to prevent crashes
df_replay_labels = df_replay_labels.dropna(subset=["playstyle", "suspicious"])
print(f"{len(df_replay_labels)} labeled replays")

1251 labeled replays


## voting logic

Every non-outlier labeled replay in a cluster votes once for each part of its `playstyle` and once for its `suspicious` value.
A cluster gets two vote counts: `technique_votes` and `suspicious_votes`.

In [4]:
def counts_dict(values):
    """counts each value and returns a plain {value: count} dict, most common first"""
    return {key: int(count) for key, count in values.value_counts().items()}

def shares_dict(values, total):
    """counts each value, turns it into a share of total and returns a plain {value: share} dict, most common first"""
    return {key: (round(count / total, 2)) for key, count in values.value_counts().items()}

def vote_cluster(df_cluster_voters):
    """takes the voters of one cluster and returns its technique shares and suspicious vote counts"""
    parts_per_voter = df_cluster_voters["playstyle"].str.split(PLAYSTYLE_SEPARATOR)
    has_ac = parts_per_voter.apply(lambda parts: "ac" in parts)
    technique_parts = parts_per_voter.explode()
    suspicious_values = df_cluster_voters.loc[~has_ac, "suspicious"].astype(int)

    voter_count = len(df_cluster_voters)
    technique_parts = technique_parts[technique_parts != "ac"]

    # strings for now, so they fit into a csv cell
    return {
        "technique_votes": str(shares_dict(technique_parts, voter_count)),
        "suspicious_votes": str(counts_dict(suspicious_values)),
    }

def label_clusters(df_export):
    """takes one cluster export and returns one row per (mode, cluster) with its votes"""
    df_voters = df_export.merge(df_replay_labels, on="idAnonymous", how="inner")
    df_voters = df_voters[~df_voters["is_outlier"]]

    rows = []
    for (mode, cluster), df_members in df_export.groupby(["mode", "cluster"]):
        df_cluster_voters = df_voters[(df_voters["mode"] == mode) & (df_voters["cluster"] == cluster)]
        row = {
            "mode": mode,
            "cluster": cluster,
            "cluster_size": len(df_members),
            "voters": len(df_cluster_voters),
        }
        row.update(vote_cluster(df_cluster_voters))
        rows.append(row)

    return pd.DataFrame(rows)

def label_replays(df_export, df_cluster_labels):
    """takes one cluster export + its cluster labels and returns one row per replay with its cluster's votes and confidence"""
    df_replays = df_export.merge(df_cluster_labels[["mode", "cluster", "technique_votes", "suspicious_votes"]], on=["mode", "cluster"], how="left")
    distance_rank = df_replays.groupby(["mode","cluster"])["distance"].rank(pct=True)
    df_replays["confidence"] = 1 - distance_rank
    return df_replays

## execute

One cell per clustering method. Each cell loads its export, votes and writes its own labels csv.

### 03b - movement & camera

In [5]:
METHOD = "movement_camera"

df_export_movement_camera = pd.read_csv(CLUSTER_EXPORT_PATHS[METHOD])
df_cluster_labels_movement_camera = label_clusters(df_export_movement_camera)
df_cluster_labels_movement_camera.to_csv(CLUSTER_LABELS_PATHS[METHOD], index=False)

df_labeled_replays_movement_camera = label_replays(df_export_movement_camera, df_cluster_labels_movement_camera)
df_labeled_replays_movement_camera.to_csv(LABELED_REPLAYS_PATHS[METHOD], index=False)
print(f"exported {len(df_labeled_replays_movement_camera)} rows to {LABELED_REPLAYS_PATHS[METHOD]}")

exported 69301 rows to ../data/notebooks_exports/labeled_replays_movement_camera.csv


### 04b - mcplayhd concept

In [6]:
METHOD = "mcplayhd_concept"

df_export_mcplayhd_concept = pd.read_csv(CLUSTER_EXPORT_PATHS[METHOD])
df_cluster_labels_mcplayhd_concept = label_clusters(df_export_mcplayhd_concept)
df_cluster_labels_mcplayhd_concept.to_csv(CLUSTER_LABELS_PATHS[METHOD], index=False)

df_labeled_replays_mcplayhd_concept = label_replays(df_export_mcplayhd_concept, df_cluster_labels_mcplayhd_concept)
df_labeled_replays_mcplayhd_concept.to_csv(LABELED_REPLAYS_PATHS[METHOD], index=False)
print(f"exported {len(df_labeled_replays_mcplayhd_concept)} rows to {LABELED_REPLAYS_PATHS[METHOD]}")

exported 69005 rows to ../data/notebooks_exports/labeled_replays_mcplayhd_concept.csv


### look-up for specific cluster labeling

In [7]:
# cell that lets us look-up how specific clusters got labeled
METHOD = "mcplayhd_concept"
check_mode = "INCLINEDSHORT"
check_cluster = 5

df_voters = df_export_mcplayhd_concept.merge(df_replay_labels, on="idAnonymous", how="inner")
df_voters = df_voters[~df_voters["is_outlier"]]
df_one = df_voters[(df_voters["mode"] == check_mode) & (df_voters["cluster"] == check_cluster)]

print(f"{check_mode} cluster {check_cluster}: {len(df_one)} voters")
display(df_one[["idAnonymous", "playstyle", "suspicious", "notes"]])

label = vote_cluster(df_one)
print(f"voted label: {label}")

INCLINEDSHORT cluster 5: 11 voters


,idAnonymous,playstyle,suspicious,notes
1047,9502,noob_noshift,1,NaN
1055,57839,noshift,0,NaN
1073,120842,ac_noshift_stair,3,NaN
1095,198575,noshift_ninja_stair,0,NaN
1121,268156,ac_noshift_stair,2,NaN
1142,309541,ac_noshift_stair,3,NaN
1155,348336,ac_noshift_stair,2,NaN
1182,420936,ac_noshift_stair,3,NaN
1190,456635,ac_noshift,3,NaN
1201,488810,ac_noshift_stair,3,NaN


voted label: {'technique_votes': "{'noshift': 1.0, 'stair': 0.64, 'ninja': 0.18, 'noob': 0.09}", 'suspicious_votes': '{0: 3, 1: 1}'}


## old voting logic (unused)

The first voting logic, which built one readable label per cluster (e.g. `technique: ninja bridge [legit]`).
Replaced by plain vote counts. Renamed to `vote_cluster_fancy` so it doesn't overwrite `vote_cluster`.

In [8]:
def vote_cluster_fancy(df_cluster_voters):
    """takes the voters of one cluster and returns its label columns"""
    count_voter = len(df_cluster_voters)
    if count_voter < MIN_LABELED_MEMBERS:
        return {"label": TOO_SMALL_LABEL}

    parts = df_cluster_voters["playstyle"].str.split(PLAYSTYLE_SEPARATOR).explode()
    votes = pd.concat([parts]).value_counts()

    def voter_share(count):
        return (count / count_voter)

    # ---------- suspicious voting ----------------
    suspicious_list = df_cluster_voters["suspicious"].tolist()

    dominant_suspicious = max(set(suspicious_list), key=suspicious_list.count)
    count_suspicious = sum(1 for n in suspicious_list if n != 0)
    count_unsuspicious = sum(1 for n in suspicious_list if n == 0)
    has_dominant_suspicious = voter_share(max(suspicious_list.count(n) for n in suspicious_list)) >= CLEAR_SHARE
    is_clearly_autoclicker = voter_share(votes.get("ac", 0)) >= CLEAR_SHARE
    is_majority_suspicious = voter_share(count_suspicious) >= MAJORITY_SHARE
    is_majority_unsuspicious = voter_share(count_unsuspicious) >= MAJORITY_SHARE

    cheat_type = "no cheat"
    if voter_share(votes.get("ac", 0)) > MAJORITY_SHARE:
        cheat_type = "autoclicker"
    if voter_share(votes.get("cheats", 0)) > MAJORITY_SHARE:
        cheat_type = "cheats"
    if voter_share(votes.get("bug", 0)) > MAJORITY_SHARE:
        cheat_type = "bugged replay"

    suspicious_label = "unclear suspicion level"
    if has_dominant_suspicious or is_clearly_autoclicker:
        suspicious_label = ""
        if cheat_type == "no cheat":
                cheat_type = "not legit"
        if np.mean(suspicious_list) > HIGHLY_SUSPICIOUS_THRESHOLD:
            suspicious_label = "highly suspicious - "
        suspicious_dict = {0: "legit", 1: "probably legit", 2: f"possibly {cheat_type}", 3: f"very likely {cheat_type}"}
        suspicious_label += suspicious_dict[dominant_suspicious]
    elif is_majority_suspicious:
        suspicious_label = ""
        if np.mean(suspicious_list) > HIGHLY_SUSPICIOUS_THRESHOLD:
            suspicious_label = "highly suspicious"
        else:
            suspicious_label += "kinda suspicious"
    elif is_majority_unsuspicious:
        suspicious_label = "probably legit"

    # ---------- technique voting -----------------
    def label_builder(clear_label_list, maybe_label_list):
        result_label = "technique: "
        maybe_joiner = ", maybe: "

        clear_label_list = sorted(clear_label_list)
        maybe_label_list = sorted(maybe_label_list)

        if len(clear_label_list) == 0:
            result_label = "no clear technique found"


        need_separator = False

        has_stair_combined = "stair_combined" in clear_label_list and len(clear_label_list) > 1
        if has_stair_combined:
            clear_label_list.remove("stair_combined")

        for item in clear_label_list:
            if need_separator:
                result_label += LABEL_JOINER
            result_label += FANCY_LABELS[item]
            need_separator = True

        if has_stair_combined:
            result_label += " " + FANCY_LABELS["stair_combined"]

        if len(maybe_label_list) > 0:
            result_label += maybe_joiner
            need_separator = False

            has_stair_combined = "stair_combined" in maybe_label_list and len(maybe_label_list) > 1
            if has_stair_combined:
                maybe_label_list.remove("stair_combined")

            for item in maybe_label_list:
                if need_separator:
                    result_label += LABEL_JOINER
                result_label += FANCY_LABELS[item]
                need_separator = True

            if has_stair_combined:
                result_label += " " + FANCY_LABELS["stair_combined"]

        return result_label


    votes_clean = votes.drop("ac", errors="ignore")
    clear_techniques = \
        [value for value, count in votes_clean.items() if voter_share(count) >= CLEAR_SHARE]
    maybe_techniques = \
        [value for value, count in votes_clean.items() if voter_share(count) > MAYBE_SHARE
         and value not in clear_techniques]

    has_clear_technique = len(clear_techniques) > 0
    has_maybe_technique = len(maybe_techniques) > 0
    clear_count = len(clear_techniques)
    maybe_count = len(maybe_techniques)

    if clear_count > 1 and "stair" in clear_techniques:
        clear_techniques = ["stair_combined" if item == "stair" else item for item in clear_techniques]

    if maybe_count > 1 and "stair" in maybe_techniques and not clear_count == 0:
        maybe_techniques = ["stair_combined" if item == "stair" else item for item in maybe_techniques]


    technique_label = "no clear technique found"
    if has_clear_technique:
        if "unclear" in clear_techniques:
            maybe_techniques.extend([item for item in clear_techniques if item != "unclear"])
            technique_label = label_builder(["unclear"], maybe_techniques)
        else:
            technique_label = label_builder(clear_techniques, maybe_techniques)
    elif has_maybe_technique:
        if "unclear" in maybe_techniques:
            maybe_techniques.remove("unclear")
        technique_label = label_builder(clear_techniques, maybe_techniques)

    full_label = f"{technique_label} [{suspicious_label}]"
    return {"label": full_label}
